# 类别、数值枚举与未知类别专项核验

2026-10-27，基于当前工作区源码。复核类别分箱、类型漂移、特殊值及训练外取值；C01-C04 已修复，下方断言验证修复后行为。修复前证据保留于 `docs/validation/categorical-binning-audit-2026-10-27.json`。

使用 `examples/hscredit_yyp.xlsx`，只保存聚合结果。

In [1]:
from pathlib import Path
from importlib import metadata
from xml.etree import ElementTree as ET
import hashlib, json, sys, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import compute_bin_stats
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'hscredit/core/binning').is_dir())
OUT=ROOT/'docs/validation';OUT.mkdir(exist_ok=True)
DATA=pd.read_excel(ROOT/'examples/hscredit_yyp.xlsx')
E={'日期':'2026-10-27','时区':'Asia/Shanghai','解释器':sys.executable,
   '依赖版本':{key:metadata.version(key) for key in ['hscredit','pandas','numpy','scikit-learn','ortools']},
   '数据行数':len(DATA),'好样本数':int(DATA.FPD.eq(0).sum()),'坏样本数':int(DATA.FPD.eq(1).sum()),
   '源码SHA256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in (ROOT/'hscredit/core/binning').glob('*.py')}}
def clean(x):
    if isinstance(x,dict): return {str(k):clean(v) for k,v in x.items()}
    if isinstance(x,(list,tuple,np.ndarray)): return [clean(v) for v in x]
    if isinstance(x,np.generic): return clean(x.item())
    if isinstance(x,float) and not np.isfinite(x): return None
    if x is pd.NA: return None
    return x

def save():
    (OUT/'categorical-binning-recheck-2026-10-27.json').write_text(json.dumps(clean(E),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')

def make(method,**overrides):
    params=dict(method=method,max_n_bins=3,min_n_bins=2,min_bin_size=.05,monotonic=False,n_jobs=1,random_state=42)
    if method in ['or_tools','cp_sat']:params.update(n_prebins=12,time_limit=5)
    if method=='genetic':params.update(population_size=12,generations=6)
    if method=='kernel_density':params.update(n_grid_points=128)
    if method in ['smooth','best_lift']:params.update(n_prebins=12)
    params.update(overrides)
    return OptimalBinning(**params)

def groups(counts,bad,values,name='类别',dtype=None):
    X=pd.DataFrame({name:pd.Series(np.repeat(values,counts),dtype=dtype)})
    y=pd.Series(np.concatenate([np.r_[np.ones(b),np.zeros(c-b)] for c,b in zip(counts,bad)]).astype(int))
    return X,y
print('样本数',len(DATA),'商品类别数',DATA['商品类别'].nunique())
print(E['依赖版本'])


样本数 970 商品类别数 6
{'hscredit': '0.1.3', 'pandas': '2.3.3', 'numpy': '2.4.6', 'scikit-learn': '1.6.1', 'ortools': '9.7.2996'}


## 1. 17 种方法的真实数据正常路径

分别使用字符串类别和统一为整数的 `category` 枚举。每种表示验证默认未知箱、严格报错、映射到已有箱三种策略（共 102 次拟合），并从原始行核对各箱人数和坏数。

In [2]:
levels=DATA['商品类别'].drop_duplicates().tolist()
codes=DATA['商品类别'].map({value:i for i,value in enumerate(levels)})
representations={'字符串类别':DATA['商品类别'].astype('string'),'整数category':codes.astype('Int64').astype('category')}
matrix=[]
for representation,series in representations.items():
    X=series.to_frame('类别')
    if representation=='字符串类别':query=pd.DataFrame({'类别':pd.Series([series.iloc[0],'核验新增类别',None],dtype='string')})
    else:query=pd.DataFrame({'类别':pd.Series([int(series.iloc[0]),999,pd.NA],dtype='Int64').astype('category')})
    for method in OptimalBinning.VALID_METHODS:
        b=make(method).fit(X,DATA.FPD)
        ids=b.transform(X)['类别'].to_numpy()
        table=compute_bin_stats(ids,DATA.FPD,round_digits=False)
        raw=pd.DataFrame({'箱':ids,'标签':DATA.FPD}).groupby('箱')['标签'].agg(['size','sum'])
        stat=table.set_index('分箱').loc[raw.index]
        np.testing.assert_array_equal(stat['样本总数'],raw['size'])
        np.testing.assert_array_equal(stat['坏样本数'],raw['sum'])
        result=b.transform(query)['类别'].tolist()
        woe=b.transform(query,metric='woe')['类别'].tolist()
        label=b.transform(query,metric='bins')['类别'].tolist()
        assert result[0]>=0 and result[1:]==[-3,-1]
        assert woe[1]==0.0 and label[1]=='unknown'
        strict=make(method,handle_unknown='raise').fit(X,DATA.FPD)
        try:
            strict.transform(query)
            raise AssertionError('未知类别未报错')
        except Exception as error:
            assert '未知类别' in str(error),str(error)
        fallback=make(method,handle_unknown=0).fit(X,DATA.FPD)
        fallback_ids=fallback.transform(query)['类别'].tolist()
        fallback_woe=fallback.transform(query,metric='woe')['类别'].tolist()
        known_woe=float(fallback.get_bin_table('类别').set_index('分箱').loc[0,'分档WOE值'])
        assert fallback_ids[1]==0 and fallback_woe[1]==known_woe
        matrix.append({'表示':representation,'方法':method,'识别类型':b.feature_types_['类别'],'实际箱数':b.n_bins_['类别'],
                       '样本数':int(stat['样本总数'].sum()),'坏样本数':int(stat['坏样本数'].sum()),
                       '未知箱号':result[1],'未知WOE':woe[1],'严格策略已报错':True,'指定箱策略正确':True})
    print('完成表示：',representation)
E['真实类别矩阵']=matrix
save()
print(pd.DataFrame(matrix).to_string(index=False))


C:\Users\Administrator\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 系统找不到指定的文件。
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Administrator\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi

C:\Users\Administrator\anaconda3\Lib\site-packages\scipy\_lib\_util.py:440: UserWarning: One of the clusters is empty. Re-run kmeans with a different initialization.
  return fun(*args, **kwargs)
C:\Users\Administrator\anaconda3\Lib\site-packages\scipy\_lib\_util.py:440: UserWarning: One of the clusters is empty. Re-run kmeans with a different initialization.
  return fun(*args, **kwargs)


C:\Users\Administrator\anaconda3\Lib\site-packages\scipy\_lib\_util.py:440: UserWarning: One of the clusters is empty. Re-run kmeans with a different initialization.
  return fun(*args, **kwargs)


完成表示： 字符串类别


C:\Users\Administrator\anaconda3\Lib\site-packages\scipy\_lib\_util.py:440: UserWarning: One of the clusters is empty. Re-run kmeans with a different initialization.
  return fun(*args, **kwargs)
C:\Users\Administrator\anaconda3\Lib\site-packages\scipy\_lib\_util.py:440: UserWarning: One of the clusters is empty. Re-run kmeans with a different initialization.
  return fun(*args, **kwargs)


C:\Users\Administrator\anaconda3\Lib\site-packages\scipy\_lib\_util.py:440: UserWarning: One of the clusters is empty. Re-run kmeans with a different initialization.
  return fun(*args, **kwargs)


完成表示： 整数category
        表示              方法        识别类型  实际箱数  样本数  坏样本数  未知箱号  未知WOE  严格策略已报错  指定箱策略正确
     字符串类别         uniform categorical     3  970   136    -3 0.0000     True     True
     字符串类别        quantile categorical     2  970   136    -3 0.0000     True     True
     字符串类别            tree categorical     3  970   136    -3 0.0000     True     True
     字符串类别             chi categorical     2  970   136    -3 0.0000     True     True
     字符串类别         best_ks categorical     3  970   136    -3 0.0000     True     True
     字符串类别         best_iv categorical     3  970   136    -3 0.0000     True     True
     字符串类别            mdlp categorical     2  970   136    -3 0.0000     True     True
     字符串类别        or_tools categorical     3  970   136    -3 0.0000     True     True
     字符串类别          cp_sat categorical     3  970   136    -3 0.0000     True     True
     字符串类别            cart categorical     3  970   136    -3 0.0000     True     True
     字符串类别          kmeans

## 2. 数值枚举必须明确类型

默认数值字段仍是连续数值；`category_order` 本身不会把它变为类别。`cat_cutoff` 是全局类型识别阈值，不是低频类别合并阈值。


In [3]:
X,y=groups([50,50,50],[40,5,25],[10,20,30],name='枚举')
results=[]
for name,train,params in [
    ('默认数值',X,{}),('显式category',X.astype({'枚举':'category'}),{}),
    ('cat_cutoff=3',X,{'cat_cutoff':3}),('仅category_order',X,{'category_order':{'枚举':[30,20,10]}})]:
    b=make('quantile',**params).fit(train,y)
    q=pd.DataFrame({'枚举':[10,40]})
    results.append({'配置':name,'识别类型':b.feature_types_['枚举'],'规则':clean(b.splits_['枚举']),
                    '已见10与未见40的箱号':b.transform(q)['枚举'].tolist()})
E['数值枚举识别']=results
print(json.dumps(results,ensure_ascii=False,indent=2))
# ordered=True 并不会代替 category_order。
ordered=pd.Series(['A','B','C']*30,dtype=pd.CategoricalDtype(['C','A','B'],ordered=True),name='类别')
b=make('quantile',min_n_bins=1).fit(ordered.to_frame(),pd.Series([0,0,1]*30))
E['有序dtype']={'声明顺序':ordered.cat.categories.tolist(),'实际顺序':b._category_orders_['类别']}
try:
    make('quantile',category_order={'类别':np.array(['A','B','C'])}).fit(ordered.to_frame(),pd.Series([0,0,1]*30))
except Exception as error:E['numpy类别顺序参数']={'异常类型':type(error).__name__,'异常':str(error)}
print(E['有序dtype'],E.get('numpy类别顺序参数'))


[
  {
    "配置": "默认数值",
    "识别类型": "numerical",
    "规则": [
      15.0,
      25.0
    ],
    "已见10与未见40的箱号": [
      0,
      2
    ]
  },
  {
    "配置": "显式category",
    "识别类型": "categorical",
    "规则": [
      [
        20
      ],
      [
        30
      ],
      [
        10
      ]
    ],
    "已见10与未见40的箱号": [
      2,
      -3
    ]
  },
  {
    "配置": "cat_cutoff=3",
    "识别类型": "categorical",
    "规则": [
      [
        20
      ],
      [
        30
      ],
      [
        10
      ]
    ],
    "已见10与未见40的箱号": [
      2,
      -3
    ]
  },
  {
    "配置": "仅category_order",
    "识别类型": "numerical",
    "规则": [
      15.0,
      25.0
    ],
    "已见10与未见40的箱号": [
      0,
      2
    ]
  }
]
{'声明顺序': ['C', 'A', 'B'], '实际顺序': ['A', 'B', 'C']} {'异常类型': 'ValueError', '异常': "特征 '类别' 的 category_order 必须返回类别序列"}


## 3. 类型漂移与未知类别信息丢失

类别匹配区分值的类型。重点比较整数、浮点、带缺失的 nullable Int64、逐批 category，以及使用固定训练类别列表的转换。


In [4]:
b=make('quantile').fit(X.astype({'枚举':'category'}),y)
nullable=pd.Series([10,40,pd.NA],dtype='Int64',name='枚举')
profiles={
    'int64无缺失':pd.Series([10,40],dtype='int64',name='枚举'),
    'float64含缺失':pd.Series([10.,40.,np.nan],name='枚举'),
    'Int64含缺失':nullable,
    'Int64转category':nullable.astype('category'),
    'Int64转object':nullable.astype('object'),
    '固定训练三种类别':pd.Series([10,40,None],dtype=pd.CategoricalDtype([10,20,30]),name='枚举'),
    '预声明四种类别':pd.Series([10,40,None],dtype=pd.CategoricalDtype([10,20,30,40]),name='枚举'),
    '字符串编码':pd.Series(['10','40',None],dtype='string',name='枚举'),
}
drift=[]
for name,values in profiles.items():
    q=values.to_frame()
    drift.append({'输入表示':name,'dtype':str(values.dtype),'实际取值':[f'{type(v).__name__}:{v!r}' for v in values.tolist()],
                  '箱号':b.transform(q)['枚举'].tolist(),'WOE':b.transform(q,metric='woe')['枚举'].tolist()})
E['预测类型与未知类别']=drift
print(json.dumps(drift,ensure_ascii=False,indent=2))
training=[]
for dtype in ['Int64','category','object']:
    values=pd.Series([10,20,30,pd.NA]*30,dtype='Int64')
    if dtype!='Int64':values=values.astype(dtype)
    try:
        model=make('quantile',cat_cutoff=3).fit(values.to_frame('枚举'),pd.Series([0,1]*60))
        ids=model.transform(values.to_frame('枚举'))['枚举']
        training.append({'类型':dtype,'训练未知箱人数':int(ids.eq(-3).sum()),'缺失人数':int(ids.eq(-1).sum()),'规则':clean(model.splits_['枚举'])})
    except Exception as error:training.append({'类型':dtype,'异常':str(error)})
E['含缺失整数枚举训练']=training
print(training)

assert all(item.get("训练未知箱人数") == 0 for item in training)
nullable_result = next(row["箱号"] for row in drift if row["输入表示"] == "Int64含缺失")
assert nullable_result[0] >= 0 and nullable_result[1:] == [-3, -1]


[
  {
    "输入表示": "int64无缺失",
    "dtype": "int64",
    "实际取值": [
      "int:10",
      "int:40"
    ],
    "箱号": [
      2,
      -3
    ],
    "WOE": [
      1.519825753744413,
      0.0
    ]
  },
  {
    "输入表示": "float64含缺失",
    "dtype": "float64",
    "实际取值": [
      "float:10.0",
      "float:40.0",
      "float:nan"
    ],
    "箱号": [
      -3,
      -3,
      -1
    ],
    "WOE": [
      0.0,
      0.0,
      0.0
    ]
  },
  {
    "输入表示": "Int64含缺失",
    "dtype": "Int64",
    "实际取值": [
      "int:10",
      "int:40",
      "NAType:<NA>"
    ],
    "箱号": [
      2,
      -3,
      -1
    ],
    "WOE": [
      1.519825753744413,
      0.0,
      0.0
    ]
  },
  {
    "输入表示": "Int64转category",
    "dtype": "category",
    "实际取值": [
      "int:10",
      "int:40",
      "float:nan"
    ],
    "箱号": [
      2,
      -3,
      -1
    ],
    "WOE": [
      1.519825753744413,
      0.0,
      0.0
    ]
  },
  {
    "输入表示": "Int64转object",
    "dtype": "object",
    "实际取值": [
      "

## 4. 特殊值身份与内部编码碰撞

输入只有字符串 A/B/C/D，配置一个根本没有出现的数值特殊码 0 或 1，本应不影响结果。断言配置前后规则完全一致。另检查 int 1、float 1.0、bool True、str "1" 的类型安全性。


In [5]:
sx,sy=groups([50]*4,[5,20,35,45],list('ABCD'))
collisions=[]
for method in ['quantile','tree','best_iv','chi','cp_sat']:
    baseline=None
    for special in [None,[0],[1],[-999]]:
        model=make(method,special_codes=special).fit(sx,sy)
        rules=clean(model.splits_['类别'])
        if special is None:baseline=rules
        collisions.append({'方法':method,'特殊码':special,'规则':rules,'与未配置特殊值一致':rules==baseline})
E['未出现特殊值碰撞']=collisions
print(pd.DataFrame(collisions).to_string(index=False))
mixed=pd.DataFrame({'混合':pd.Series([1,1.0,True,'1','A']*20,dtype=object)})
my=pd.Series([0,1]*50)
identity=[]
for special in [None,[1]]:
    model=make('quantile',max_n_bins=5,min_n_bins=1,special_codes=special).fit(mixed,my)
    identity.append({'特殊码':special,'输入':[f'{type(v).__name__}:{v!r}' for v in mixed['混合'].iloc[:5]],
                     '箱号':model.transform(mixed.iloc[:5])['混合'].tolist(),
                     '标签':model.transform(mixed.iloc[:5],metric='bins')['混合'].tolist()})
E['混合类型特殊值']=identity
print(identity)

assert all(row["与未配置特殊值一致"] for row in collisions)
assert identity[1]["箱号"][0] == -2
assert all(value >= 0 for value in identity[1]["箱号"][1:])


      方法    特殊码                 规则  与未配置特殊值一致
quantile   None [[A], [B], [C, D]]       True
quantile    [0] [[A], [B], [C, D]]       True
quantile    [1] [[A], [B], [C, D]]       True
quantile [-999] [[A], [B], [C, D]]       True
    tree   None [[A], [B], [C, D]]       True
    tree    [0] [[A], [B], [C, D]]       True
    tree    [1] [[A], [B], [C, D]]       True
    tree [-999] [[A], [B], [C, D]]       True
 best_iv   None [[A], [B], [C, D]]       True
 best_iv    [0] [[A], [B], [C, D]]       True
 best_iv    [1] [[A], [B], [C, D]]       True
 best_iv [-999] [[A], [B], [C, D]]       True
     chi   None [[A], [B], [C, D]]       True
     chi    [0] [[A], [B], [C, D]]       True
     chi    [1] [[A], [B], [C, D]]       True
     chi [-999] [[A], [B], [C, D]]       True
  cp_sat   None [[A], [B, C], [D]]       True
  cp_sat    [0] [[A], [B, C], [D]]       True
  cp_sat    [1] [[A], [B, C], [D]]       True
  cp_sat [-999] [[A], [B, C], [D]]       True
[{'特殊码': None, '输入': ['int:1', 'fl

## 5. 手工类别规则与 WOE 一致性

分别验证正常 fit、`user_splits` 固定规则、`import_rules(...).fit(...)`、携带 WOE 的 export/load。只导入规则后拟合，不应把占位 WOE 当成已经学习过的 WOE。


In [6]:
train=X.astype({'枚举':'category'})
known=pd.DataFrame({'枚举':pd.Series([10,20,30],dtype='category')})
manual=make('quantile',user_splits={'枚举':[[10],[20],[30]]},user_splits_fixed=True).fit(train,y)
imported=make('quantile').import_rules({'枚举':[[10],[20],[30],[40]]}).fit(train,y)
loaded=make('quantile').load(manual.export())
rows=[]
for name,model in [('固定用户规则',manual),('只导入规则后fit',imported),('完整export/load',loaded)]:
    transformed=model.transform(known,metric='woe')['枚举'].tolist()
    table=model.bin_tables_.get('枚举')
    stored=None if table is None else table[['分箱','分档WOE值']].to_dict('records')
    rows.append({'入口':name,'训练表WOE':stored,'已见类别实际转换WOE':transformed})
np.testing.assert_allclose(manual.transform(known,metric='woe'),loaded.transform(known,metric='woe'))
E['规则导入WOE一致性']=rows
print(json.dumps(clean(rows),ensure_ascii=False,indent=2))

bad = np.array([40, 5, 25], dtype=float)
good = 50 - bad
expected_woe = np.log((bad / bad.sum()) / (good / good.sum()))
np.testing.assert_allclose(imported.transform(known, metric="woe")["枚举"], expected_woe)


[
  {
    "入口": "固定用户规则",
    "训练表WOE": [
      {
        "分箱": 0,
        "分档WOE值": 1.519825753744413
      },
      {
        "分箱": 1,
        "分档WOE值": -2.063693184711697
      },
      {
        "分箱": 2,
        "分档WOE值": 0.13353139262452257
      }
    ],
    "已见类别实际转换WOE": [
      1.519825753744413,
      -2.063693184711697,
      0.13353139262452257
    ]
  },
  {
    "入口": "只导入规则后fit",
    "训练表WOE": [
      {
        "分箱": 0,
        "分档WOE值": 1.519825753744413
      },
      {
        "分箱": 1,
        "分档WOE值": -2.063693184711697
      },
      {
        "分箱": 2,
        "分档WOE值": 0.13353139262452257
      }
    ],
    "已见类别实际转换WOE": [
      1.519825753744413,
      -2.063693184711697,
      0.13353139262452257
    ]
  },
  {
    "入口": "完整export/load",
    "训练表WOE": null,
    "已见类别实际转换WOE": [
      1.519825753744413,
      -2.063693184711697,
      0.13353139262452257
    ]
  }
]


## 6. 高基数文本的边界

字符串在这里作为完整枚举匹配，不做分词或语义理解。唯一文本可以通过训练标签排序制造完美的训练分离，而新文本全部落入未知箱；这不是现实数据的效果评估，只是解释该机制的合成反例。小规模耗时用于观察类别数量影响，不代表百万行容量验收。


In [7]:
count=120
ids=pd.DataFrame({'文本':[f'记录_{i}' for i in range(count)]})
labels=pd.Series([0,1]*(count//2))
model=make('tree',max_n_bins=2).fit(ids,labels)
unseen=pd.DataFrame({'文本':[f'新记录_{i}' for i in range(count)]})
E['唯一文本反例']={'训练AUC':float(roc_auc_score(labels,model.transform(ids,metric='woe')['文本'])),
    '未见文本AUC':float(roc_auc_score(labels,model.transform(unseen,metric='woe')['文本'])),
    '未见文本未知率':float(model.transform(unseen)['文本'].eq(-3).mean())}
print(E['唯一文本反例'])
timings=[]
for cardinality in [10,100,200]:
    frame=pd.DataFrame({'类别':[f'类别_{i%cardinality}' for i in range(1000)]})
    target=pd.Series([0,1]*500)
    start=time.perf_counter()
    model=make('uniform',min_bin_size=1).fit(frame,target)
    elapsed=time.perf_counter()-start
    timings.append({'行数':1000,'唯一类别数':cardinality,'拟合耗时秒':elapsed})
E['小规模类别耗时']=timings
print(pd.DataFrame(timings).to_string(index=False))


{'训练AUC': 1.0, '未见文本AUC': 0.5, '未见文本未知率': 1.0}

  行数  唯一类别数  拟合耗时秒
1000     10 0.0865
1000    100 0.7241
1000    200 1.4232


## 7. 验证记录

保留当前源码指纹、实际解释器和本轮类别回归范围；通过数不与其他重叠测试相加。

In [8]:
xml=ROOT/'.audit_tmp/prepush-category-final.xml'
if xml.exists():
    suite=ET.parse(xml).getroot().find('testsuite')
    E['修复后类别回归']={'日期':'2026-10-27','解释器':sys.executable,
       '范围':['test_categorical_audit_repairs.py','test_categorical_adapter.py','test_categorical_methods.py','test_categorical_rules.py','test_categorical_binning_complete.py','test_binning_contracts.py'],
       **{k:int(suite.get(k,0)) for k in ['tests','failures','errors','skipped']},'耗时秒':float(suite.get('time'))}
    assert E['修复后类别回归']['failures']==E['修复后类别回归']['errors']==0
assert len(E['真实类别矩阵'])==34
for name,digest in E['源码SHA256'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==digest,name
E['核验已完成']=True
save()
print('结果文件：',OUT/'categorical-binning-recheck-2026-10-27.json')


结果文件： E:\workspace\hscredit\.audit_tmp\prepush-notebook-evidence\categorical-binning-recheck-2026-10-27.json
